## 1. Foundations and Model A: steady populations under a fixed catch

**Question.** With the same catch every year, where can the population stay steady, which equilibrium points are stable, and what is the largest catch before collapse is certain?

**Model A.** A well-mixed population $N(t)$ with growth rate $r$, carrying capacity $K$ and Allee threshold $A$ ($0<A<K$), harvested at a constant rate $H$:
$$\frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-H.$$
Below $A$ growth is negative, so a small population shrinks even without fishing. We assume deterministic dynamics and constant parameters, and stop a simulation when $N$ reaches 0, since an extinct stock cannot be fished.

**Scaling.** With $x=N/K$, $\tau=rt$, $a=A/K$ and $q=H/(rK)$,
$$\frac{dx}{d\tau}=f(x)=G(x)-q,\qquad G(x)=x(1-x)\Big(\frac{x}{a}-1\Big),$$
so only $a$ and $q$ matter. We use $a=0.2$, for example $r=0.5$/year, $K=1000$ t and $A=200$ t as in Section 2.

### 1.1 Steady states and stability

A steady state $x_e$ solves $f(x_e)=0$. It is **stable** if $f>0$ just below it and $f<0$ just above it, so a small disturbance dies out. With the opposite signs it is **unstable**, and with the same sign on both sides it is **semistable**.

**Logistic baseline** ($f=x(1-x)-q$, no Allee effect). For $q<\tfrac14$ the steady states $x_\pm=\tfrac12\big(1\pm\sqrt{1-4q}\big)$ are unstable (lower) and stable (upper). They merge at $q=\tfrac14$, so the largest catch is $H_c=rK/4$. Without harvest, zero is unstable and any small population recovers.

In [ ]:
import matplotlib.pyplot as plt

from mm_allee.equilibria import equilibria_logistic

for q in [0.0, 0.1, 0.25, 0.3]:
    states = [f"x={e.x:.3f} ({e.stability})" for e in equilibria_logistic(q)]
    print(f"q={q}:", ", ".join(states) or "none, the population collapses")

### 1.2 Model A: the growth curve and the maximum catch

$G$ is negative on $(0,a)$ and forms a hill on $(a,1)$. Setting $G'(x)=0$ gives the top of the hill,
$$x^*=\frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x^*).$$
Steady states are where the line at height $q$ meets $G$:

- **$q=0$:** $0$ (stable), $a$ (unstable, the minimum viable population) and $1$ (stable).
- **$0<q<q_c$:** $x_-$ between $a$ and $x^*$ (unstable) and $x_+$ between $x^*$ and 1 (stable). Below $x_-$ the population dies out; above it, it settles at $x_+$.
- **$q=q_c$:** one semistable point at $x^*$, the fold.
- **$q>q_c$:** none, so every population collapses.

For $q>0$, $f(0)=-q<0$, so zero is  no longer an equilibrium point. If the initial population is below the unstable equilibrium (x_-), it decreases and reaches zero in finite time. For $a=0.2$, $x^*\approx0.706$ and $q_c\approx0.525$, a maximum catch of about 263 t/year in the example.

### 1.3 Finding the steady states numerically

**Algorithm.**
1. Compute $x^*$ and $q_c$.
2. If $q>q_c$ there is no positive steady state; if $q=q_c$ there is one, at $x^*$.
3. If $0<q<q_c$, $f$ changes sign once on $[a,x^*]$ and once on $[x^*,1]$. Find each root with Brent's method, a standard root finder that narrows an interval around a sign change.
4. Label each root from the signs of $f$ on either side, as in 1.1.

In [ ]:
from mm_allee.equilibria import equilibria_a, fold_a

a = 0.2
x_star, q_c = fold_a(a)
print(f"x* = {x_star:.4f}, q_c = {q_c:.4f}\n")
for q in [0.0, 0.3, q_c, 0.6]:
    states = [f"x={e.x:.4f} ({e.stability})" for e in equilibria_a(a, q)]
    print(f"q={q:.4f}:", ", ".join(states) or "none, the population collapses")

### 1.4 Growth curves and the branch diagram

Filled circles are stable, open circles unstable and half-filled semistable. Arrows on the phase lines show whether the population grows or shrinks.

In [ ]:
from mm_allee.equilibria_plots import branch_figure, perturbation_figure, production_figure

fig = production_figure(a=0.2, logistic_q=(0.1, 0.25, 0.35), allee_q=(0.25 * q_c, q_c, 1.2 * q_c))
plt.show()

**Interpretation.** As the constant harvesting rate increases, the stable equilibrium (solid curve) and the unstable equilibrium (dashed curve), which acts as the collapse threshold, move closer together. They eventually meet at the critical point $(q_c, x^*)$. Beyond $q_c$, no positive equilibrium exists, and the population eventually becomes extinct.

In [ ]:
fig = branch_figure(a=0.2)
plt.show()

**Interpretation.** As the catch rises, the stable branch (solid) and the collapse threshold (dashed) move together and meet at the fold $(q_c,x^*)$, the gap between the equilibrium points decreases as the harvesting rate approaches its critical value.

### 1.5 Checking stability by simulation

At $q=0.3$, start $\delta=0.01$ above and below each steady state and solve the ODE (RK45, stopping at zero).

In [ ]:
fig = perturbation_figure(a=0.2, q=0.3, delta=0.01, tau_end=12.0)
plt.show()

Both starts near $x_+$ return to it. Just above $x_-$ the population recovers, and just below it reaches zero at $\tau\approx2.9$, as the stability labels predict.

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Main result</strong></div>

As the constant harvesting rate increases, the stable and unstable equilibrium points move closer together and eventually merge at the critical harvesting rate (q_c). A population starting near the stable equilibrium can still collapse if a sufficiently large decrease takes it below the unstable equilibrium. Therefore, harvesting close to (q_c) leaves the population more vulnerable to disturbances.

**Limitations.** No randomness (which matters most near the threshold), a simple cubic growth law, constant parameters, and an exact catch whenever $N>0$. The numbers are illustrative, not from a real stock.